In [9]:
import os
os.environ["OLLAMA_HOST"] = "http://localhost:11300" 
print(os.environ.get("OLLAMA_HOST"))

http://localhost:11300


In [ ]:
pip install -r requirements.txt

In [11]:
import Prompts.ver_2 as prompts
import importlib
importlib.reload(prompts)
from Prompts.ver_2 import Config
import evaluate
importlib.reload(evaluate)

<module 'evaluate' from '/home/aluc31or/LLM_Project/evaluate.py'>

In [12]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [13]:
def evaluate_model(df, results):
    # Get true labels from the dataframe
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [14]:
def run_prompt_eval(dataset_path, prompt_template, model, text_col="text"):
    
    df_local = pd.read_csv(dataset_path)
    reviews = df_local[text_col].tolist()

    prompt = PromptTemplate(
        input_variables=["text"],
        template=prompt_template,
    )

    llm = OllamaLLM(model=model)
    chain = prompt | llm

    predictions = []
    for review in reviews:
        response = chain.invoke({"text": review})
        label = response.strip().lower()

        if "fake" in label:
            cleaned_label = "fake"
        elif "real" in label:
            cleaned_label = "real"
        else:
            cleaned_label = "fake"  # safety fallback
        
        cleaned_label = cleaned_label.strip('.,!?;:')

        #print(f"Raw output: {response} -> Cleaned: {cleaned_label}")
        predictions.append(cleaned_label)

    accuracy, f1, conf_matrix = evaluate_model(df_local, predictions)
    return df_local, predictions, accuracy, f1, conf_matrix

# 1. Amazon Human vs Computer

In [15]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,source,text,is_synthetic
0,fake,Home_and_Kitchen,amazon,Air pressure was utter crap. I kept the lid o...,1
1,fake,Home_and_Kitchen,amazon,"Not what I expected, smaller than expected, bu...",1
2,fake,Home_and_Kitchen,amazon,We like a toaster that doesn't have the long h...,1
3,fake,Home_and_Kitchen,amazon,Love these pans they work great. The only prob...,1
4,fake,Home_and_Kitchen,amazon,I love the print on the back and the finish. I...,1


## 1.1 Zero Shot Prompting

In [ ]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [ ]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.7375

F1 Score: 0.7374852335443869

Confusion Matrix:
[[298 102]
 [108 292]]


## 1.2 One-Shot Prompting

In [16]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b"
 )

In [17]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.6925

F1 Score: 0.6724345701369516

Confusion Matrix:
[[376  24]
 [222 178]]


## 1.3 Few-Shot Prompting

In [18]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="llama3:8b"
 )

In [19]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.71

F1 Score: 0.6966209854587301

Confusion Matrix:
[[368  32]
 [200 200]]


# 2. Hotel Human Real vs Human Fake

In [20]:
from Prompts.ver_2 import Config
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews. Carefully analyze the following review for signs such as:
    - Overly generic or vague language
    - Exaggerated praise or criticism
    - Repetitive or templated phrasing
    - Marketing-like wording or unnatural flow
    - Specific details vs. general statements
    Review: "{text}"
    Classify this review as either "real" or "fake" (respond with only one word):
    


In [21]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,truthful,conrad,Hotel,We stayed for a one night getaway with family ...,0,TripAdvisor
1,truthful,hyatt,Hotel,Triple A rate with upgrade to view room was le...,0,TripAdvisor
2,truthful,hyatt,Hotel,This comes a little late as I'm finally catchi...,0,TripAdvisor
3,truthful,omni,Hotel,The Omni Chicago really delivers on all fronts...,0,TripAdvisor
4,truthful,hyatt,Hotel,I asked for a high floor away from the elevato...,0,TripAdvisor


## 2.1 Zero Shot Prompting

In [ ]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [ ]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5992462311557789

F1 Score: 0.5788925539518832

Confusion Matrix:
[[302 494]
 [144 652]]


## 2.2 One-Shot Prompting

In [22]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b")

In [ ]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5640703517587939

F1 Score: 0.5192692932242889

Confusion Matrix:
[[206 590]
 [104 692]]


## 2.3 Few-Shot Prompting

In [23]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="llama3:8b"
 )

In [ ]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.6030150753768844

F1 Score: 0.5751081081081082

Confusion Matrix:
[[276 520]
 [112 684]]


# 3. Human Real vs CG

In [24]:
from Prompts.ver_2 import Config
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews. Carefully analyze the following review for signs such as:
    - Overly generic or vague language
    - Exaggerated praise or criticism
    - Repetitive or templated phrasing
    - Marketing-like wording or unnatural flow
    - Specific details vs. general statements
    Review: "{text}"
    Classify this review as either "real" or "fake" (respond with only one word):
    


In [25]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,real,james,Hotel,My husband and I were there for a conference a...,0,Web
1,real,palmer,Hotel,Here are my experiences: - Had three rooms res...,0,Web
2,real,omni,Hotel,My wife and I travelled to Chicago and really ...,0,TripAdvisor
3,real,knickerbocker,Hotel,The location of this hotel is excellent. The h...,0,Web
4,real,fairmont,Hotel,We recently completed our second stay at the F...,0,TripAdvisor


## 3.1 Zero Shot Prompting

In [ ]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [ ]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5012562814070352

F1 Score: 0.447159364257411

Confusion Matrix:
[[150 646]
 [148 648]]


## 3.2 One-Shot Prompting

In [26]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b"
 )

In [ ]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.7317839195979899

F1 Score: 0.7269881278406776

Confusion Matrix:
[[477 319]
 [108 688]]


## 3.3 Few-Shot Prompting

In [27]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="llama3:8b"
 )

In [ ]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.8128140703517588

F1 Score: 0.8123865024075726

Confusion Matrix:
[[609 187]
 [111 685]]


# 4. Human Real vs MixFake

In [30]:
from Prompts.ver_2 import Config
print(Config.zero_shot_prompt_template)


    You are an expert at detecting fake and real reviews. Carefully analyze the following review for signs such as:
    - Overly generic or vague language
    - Exaggerated praise or criticism
    - Repetitive or templated phrasing
    - Marketing-like wording or unnatural flow
    - Specific details vs. general statements
    Review: "{text}"
    Classify this review as either "real" or "fake" (respond with only one word):
    


In [31]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,real,sofitel,Hotel,Just got back from a 10 day visit to Chicago. ...,0,TripAdvisor
1,real,palmer,Hotel,Just returned from a week in Chicago with the ...,0,TripAdvisor
2,real,hardrock,Hotel,Not worth the price. We almost stayed at a hot...,0,Web
3,real,homewood,Hotel,WARNING: FOOD POISONING ALERT!!! The room was ...,0,Web
4,real,ambassador,Hotel,Upon check in - the lobby was BUSY! I was tire...,0,TripAdvisor


## 4.1 Zero Shot Prompting

In [32]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.zero_shot_prompt_template,
    model="llama3:8b"
 )

In [33]:
print("=" * 50)
print("zero_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.5559045226130653

F1 Score: 0.5258197983454265

Confusion Matrix:
[[242 554]
 [153 643]]


## 4.2 One-Shot Prompting

In [34]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.one_shot_prompt_template,
    model="llama3:8b"
 )

In [35]:
print("=" * 50)
print("one_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.7141959798994975

F1 Score: 0.713987321404951

Confusion Matrix:
[[590 206]
 [249 547]]


## 4.3 Few-Shot Prompting

In [36]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.few_shot_prompt_template,
    model="llama3:8b"
 )

In [37]:
print("=" * 50)
print("few_shot_prompt_template Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_prompt_template Results
Model: llama3:8b
Accuracy: 0.7361809045226131

F1 Score: 0.7358540655339805

Confusion Matrix:
[[558 238]
 [182 614]]
